In [1]:
import numpy as np
import xarray as xr

In [2]:
filepath = 'C:/Users\Ian\projects\python3-crlx\__scratch__/test_data\SOCATv2024_tracks_gridded_monthly.nc'
filepath2 = 'C:/Users\Ian\projects\python3-crlx\__scratch__/test_data\SOCATv2024_qrtrdeg_gridded_coast_monthly.nc'

In [3]:
ds = xr.open_dataset(filepath)
ds = ds.rename({'tmnth':'time','xlon':'longitude','ylat':'latitude'})
ds = ds.rename({'salinity_min_unwtd': 'sal_min', 'salinity_max_unwtd': 'sal_max',
                'fco2_min_unwtd': 'fco2_min', 'fco2_max_unwtd': 'fco2_max'})
ds = ds[['fco2_min','fco2_max','sal_min','sal_max']]
ds = ds.groupby('time.month').mean()

In [4]:
ds2 = xr.open_dataset(filepath2)
ds2 = ds2.rename({'tmnth':'time','xlon':'longitude','ylat':'latitude'})
ds2 = ds2.rename({'coast_salinity_min_unwtd': 'sal_min', 'coast_salinity_max_unwtd': 'sal_max',
                'coast_fco2_min_unwtd': 'fco2_min', 'coast_fco2_max_unwtd': 'fco2_max'})
ds2 = ds2[['fco2_min','fco2_max','sal_min','sal_max']]
ds2 = ds2.sel(latitude = ds.latitude, longitude = ds.longitude, method = 'nearest')
ds2['latitude'] = ds.latitude
ds2['longitude'] = ds.longitude
ds2 = ds2.groupby('time.month').mean()

In [5]:
ds3 = (ds + ds2)/2
ds3.attrs['dataset'] = 'SOCATv2024'
ds3.attrs['creator'] = 'Ian Black'
ds3.attrs['creator_description'] = 'This reference dataset combines SOCAT global and coastal data on 1x1 degree grids. Coastal data were coarsened from 0.25 to 1 degrees. Data consist of the unweighted minimums and maximums for the respective variables. This data should not be used to flag data as bad or poor.'
ds3.attrs['spatial_resolution'] = 1
ds3.attrs['spatial_resolution_units'] = 'degrees'
ds3.attrs['temporal_resolution'] = 'month'

In [6]:
filename = 'test_data/socat2024_monthly_minmax_1970-2024.nc'
ds3.to_netcdf(filename, engine = 'netcdf4')